# GeoXplain Aurora adapter quickstart

Needs [`geoxplain-aurora-adapter`](https://github.com/clemenskoprolin/geoxplain-aurora-adapter) to be installed either locally or on a remote HPC server.

## How to run

Start it it in listener mode (`geoxplain-aurora-adapter listen`) on the remote server.

If this notebook already runs inside a GPU environment, set `REMOTE = None` or remove the `remote=REMOTE` arguments to run in-process.

In [ ]:
import geoxplain_aurora_adapter as ax
from geoxplain import GeoXplainWidget

REMOTE = "http://localhost:8765"
# REMOTE = None  # use this when running directly in a GPU notebook


# Call a method

Construct a target, pass `remote=REMOTE`, and hand the result straight to the widget.

In [ ]:
target = ax.Target.box(
    var="q", level=850,
    lat=47.38, lon=8.54,  # Zurich
    timestamp="2024-07-15T12:00:00Z",
    size=(1.0, 1.6),
)

result = ax.run_ig(
    target=target,
    input=["q", "t"],
    levels=[925, 850, 700],
    remote=REMOTE,
)

w = GeoXplainWidget(result=result, height=620)
w


# Add a matching weather overlay

No date is passed here. The adapter remembers the timestamp from the previous XIA request and asks the compute side for the matching weather field.

In [ ]:
humidity_overlay = ax.pull_overlay(
    "q",
    level=850,
    remote=REMOTE,
)

w.add_overlay(humidity_overlay)


# Bundle multiple timeframes

`timeframes` returns one `.xia` result bundle with all frames on the widget timeline. This example attributes one surface input variable to keep the bundle compact.

In [ ]:
bundle = ax.run_saliency(
    target=target,
    input=["2t"],
    remote=REMOTE,
    timeframes=5,
    step_hours=6,
)

w_bundle = GeoXplainWidget(result=bundle, height=620)
w_bundle


# Request rollout

The rollout API uses the same target and input structure, with an explicit method and timeframe count.

In [ ]:
rollout = ax.run_rollout(
    target=target,
    input=["q"],
    method="saliency",  # saliency and ig are currently supported for rollout
    timeframes=2,
    remote=REMOTE,
)

w_rollout = GeoXplainWidget(result=rollout, height=620)
w_rollout
